## Concept focus — Generator-based coroutines and suspension

This exercise connects generators to coroutine thinking. Understanding `send`, `throw`, and `close` makes async code feel less magical because you can see how execution pauses and resumes around a suspension point.

```text
caller --send(value)--> generator paused at yield
caller <--yield(result)-- generator resumes later

paused frame keeps:
- local variables
- current instruction
- exception handling state
```

### How to think about it
A coroutine is best understood as a live paused frame. Instead of thinking only “this returns values,” think “this function can stop mid-flight, keep its internal state, then continue from the exact same place later.”

### Visual references and further study
- [PEP 342 — coroutines via enhanced generators](https://peps.python.org/pep-0342/)
- [contextlib documentation](https://docs.python.org/3/library/contextlib.html)
- [Python Tutor visualizer](https://pythontutor.com/visualize.html)
- [Generator introduction by Beazley](https://www.dabeaz.com/generators/)

---

# Module 14 — Iterators, Generators, and Lazy Pipelines

## Exercise 14.4 — Generators that receive

Run:  python ex04_coroutines.py

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.

---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 1. The iterator protocol

Two methods, and everything else in this module is built on them.

In [ ]:
iter(obj)      # -> obj.__iter__()  : returns an ITERATOR
next(it)       # -> it.__next__()   : the next value, or raises StopIteration

A `for` loop is sugar for exactly this:

In [ ]:
for x in things: process(x)

# is precisely:
it = iter(things)
while True:
    try:
        x = next(it)
    except StopIteration:
        break
    process(x)

| | Iterable | Iterator |
|---|---|---|
| Defines | `__iter__` | `__iter__` **and** `__next__` |
| `__iter__` returns | a **fresh** iterator | `self` |
| Reusable | yes | **no** |
| Examples | `list`, `dict`, `str`, `range` | generators, file objects, `iter([])` |

**Every iterator is an iterable; not every iterable is an iterator.** The
distinction shows up as the one-shot bug (Module 09), and it is worth being able
to state precisely:

In [ ]:
data = (x for x in range(3))
list(data)      # [0, 1, 2]
list(data)      # []          <- exhausted, silently

No error. That silence is the whole hazard.

---

## Concept 2. Generator functions

Any function containing `yield` is a generator function. Calling it **runs
nothing** — it returns a generator object.

In [ ]:
def countdown(n: int):
    print("starting")            # does NOT run on the call
    while n > 0:
        yield n
        n -= 1
    print("done")

gen = countdown(3)               # nothing printed
next(gen)                         # 'starting', then 3
next(gen)                         # 2

`yield` **suspends** the function: locals, instruction pointer, and the whole
frame are preserved. `next()` resumes exactly where it stopped. That suspended
frame is the mental image to carry — it is also how `await` works (Module 22).

### Generators are the easiest way to write `__iter__`

Compare this with Module 09's iterator class:

In [ ]:
class Countdown:
    def __init__(self, start: int) -> None:
        self.start = start

    def __iter__(self):
        current = self.start      # a LOCAL, so each call gets fresh state
        while current > 0:
            yield current
            current -= 1

Two `for` loops both work, because each call to `__iter__` creates a new
generator with its own locals. That is the fix for the one-shot bug, and it is
free.

### `yield from`

In [ ]:
def flatten(nested):
    for item in nested:
        if isinstance(item, list):
            yield from flatten(item)      # delegate, recursively
        else:
            yield item

`yield from x` is not just `for i in x: yield i` — it also forwards `send`,
`throw` and `close`, and propagates the sub-generator's return value. For plain
iteration the loop is equivalent; for coroutines it is not.

### Generator expressions

In [ ]:
squares = (x * x for x in range(1_000_000))     # lazy, ~200 bytes
squares = [x * x for x in range(1_000_000)]     # eager, ~40 MB

sum(x * x for x in data)                         # parens optional as sole arg
any(line.startswith("ERROR") for line in fh)     # short-circuits

**Use a generator expression when the values are consumed once.** Use a list
when you need to index, re-iterate, or take `len()`.

---

## Concept 3. Pipelines

The technique that makes this module worth its time. Each stage is lazy; the
data flows through one item at a time.

In [ ]:
def read_lines(path):
    with open(path, encoding="utf-8") as fh:
        yield from fh

def parse(lines):
    for line in lines:
        parts = line.rstrip("\n").split("\t")
        if len(parts) == 4:
            yield {"ts": parts[0], "level": parts[1],
                   "user": parts[2], "msg": parts[3]}

def only(records, level):
    for r in records:
        if r["level"] == level:
            yield r

def summarise(records, limit):
    for r in islice(records, limit):
        yield f"{r['ts']} {r['user']}: {r['msg']}"

# nothing has run yet
pipeline = summarise(only(parse(read_lines("50gb.log")), "ERROR"), 10)

for line in pipeline:      # NOW it runs, one line at a time
    print(line)

Memory: one line. Work done: it stops after finding ten errors, even if the file
is 50 GB and the tenth error is on line 900.

**Three properties that fall out:**

1. **Constant memory**, regardless of input size.
2. **Early termination** — `break` at any point stops all upstream work.
3. **Composability** — any stage can be inserted, removed, or reordered without
   touching the others.

### The `with` trap in a generator

In [ ]:
def read_lines(path):
    with open(path) as fh:
        yield from fh          # the file stays open while the generator lives

If the consumer abandons the generator, the `with` block exits when the
generator is garbage collected — which is *usually* immediate under CPython
refcounting and *not guaranteed* (Module 02). For long-lived programs, close it
explicitly or use `contextlib.closing`. This is a real source of "too many open
files" in production.

---

## Concept 5. Generators as coroutines

`yield` is an expression, so a generator can *receive* values.

In [ ]:
def averager():
    total, count = 0.0, 0
    average = None
    while True:
        value = yield average        # RECEIVES from send(), yields the average
        total += value
        count += 1
        average = total / count

avg = averager()
next(avg)              # "prime" it: run to the first yield
avg.send(10)           # 10.0
avg.send(20)           # 15.0

Three methods drive a generator from outside:

In [ ]:
gen.send(value)        # resume, with `value` as the result of the yield
gen.throw(SomeError)   # raise inside the generator at the yield point
gen.close()            # raise GeneratorExit at the yield point

This is where `async`/`await` came from historically — before native
coroutines, `asyncio` was built on `yield from` over generators. You will rarely
write `send()` today, but understanding it makes Module 22 straightforward
rather than mysterious.

`contextlib.contextmanager` is the one place you use this daily:

In [ ]:
@contextmanager
def managed():
    setup()
    try:
        yield resource        # the with-block runs HERE, at the suspension
    finally:
        teardown()

The generator suspends at `yield`, the `with` body runs, and then the generator
is resumed to run its `finally`. `__exit__` is implemented by calling `send` or
`throw` on it — a direct application of everything above.

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: The iterator protocol
- Section 2: Generator functions
- Section 3: Pipelines
- Section 4: `itertools`
- Section 5: Generators as coroutines
- Section 6: When *not* to be lazy

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

from collections.abc import Callable, Generator, Iterator
from typing import Any


# TODO 1 -----------------------------------------------------------------------

---

## `running_stats`

A coroutine that receives numbers and yields running statistics.

In [ ]:
def running_stats() -> Generator[dict[str, float], float, None]:
    """A coroutine that receives numbers and yields running statistics.

        s = running_stats(); next(s)
        s.send(10)  -> {"count": 1, "mean": 10.0, "min": 10, "max": 10, ...}

    Include a running standard deviation using Welford's algorithm -- computing
    it from a stored list defeats the point, which is CONSTANT memory over an
    unbounded stream.
    """
    raise NotImplementedError

---

## `prime`

A decorator that calls next() on a new coroutine automatically.

In [ ]:
def prime(fn: Callable[..., Generator]) -> Callable[..., Generator]:  # type: ignore[type-arg]
    """A decorator that calls next() on a new coroutine automatically.

    Forgetting to prime gives 'TypeError: can't send non-None value to a
    just-started generator', which is confusing the first three times. This
    decorator is the standard fix, and it is a preview of Module 15.
    """
    raise NotImplementedError

---

## `broadcast`

Fan out: everything sent here is forwarded to every target coroutine.

In [ ]:
def broadcast(*targets: Generator[Any, Any, None]) -> Generator[None, Any, None]:
    """Fan out: everything sent here is forwarded to every target coroutine.

    Requirements:
      - a target that raises must not stop the others
      - close() must close every target
      - the first target must not be able to modify what the second receives
    """
    raise NotImplementedError

---

## `my_contextmanager`

Implement contextlib.contextmanager from scratch.

In [ ]:
def my_contextmanager(fn):  # type: ignore[no-untyped-def]
    """Implement contextlib.contextmanager from scratch.

    The whole mechanism, and it is a genuinely beautiful piece of design:
      __enter__  = next(gen), returning the yielded value
      __exit__   = gen.throw(exc) if the block raised, else next(gen)
                   and StopIteration means the generator finished normally

    Handle correctly:
      - the block raising -> throw INTO the generator so its finally runs
      - the generator suppressing the exception (yield inside try/except that
        does not re-raise) -> __exit__ returns True
      - the generator yielding twice -> RuntimeError("generator didn't stop")
      - the generator not yielding at all -> RuntimeError

    Then answer: why must __exit__ use throw() rather than simply calling
    next()? What would a `finally` in the generator do in each case?
    """
    raise NotImplementedError

---

## `verify`

_verify_

In [ ]:
def verify() -> None:
    s = running_stats()
    next(s)
    s.send(10)
    result = s.send(20)
    assert result["count"] == 2 and result["mean"] == 15.0
    assert result["min"] == 10 and result["max"] == 20

    @prime
    def collector() -> Generator[list[Any], Any, None]:
        items: list[Any] = []
        while True:
            items.append((yield items))

    c = collector()
    assert c.send("a") == ["a"]            # no manual next() needed

    seen_a: list[Any] = []
    seen_b: list[Any] = []

    @prime
    def recorder(into: list[Any]) -> Generator[None, Any, None]:
        while True:
            into.append((yield))

    @prime
    def exploder() -> Generator[None, Any, None]:
        while True:
            yield
            raise RuntimeError("bad target")

    b = broadcast(recorder(seen_a), exploder(), recorder(seen_b))
    b.send("x")
    b.send("y")
    assert seen_a == ["x", "y"], seen_a
    assert seen_b == ["x", "y"], "a raising target broke the broadcast"

    log: list[str] = []

    @my_contextmanager
    def managed(name: str):  # type: ignore[no-untyped-def]
        log.append(f"enter {name}")
        try:
            yield name.upper()
        finally:
            log.append(f"exit {name}")

    with managed("db") as handle:
        assert handle == "DB"
        log.append("body")
    assert log == ["enter db", "body", "exit db"], log

    log.clear()
    try:
        with managed("db"):
            raise ValueError("boom")
    except ValueError:
        pass
    assert log == ["enter db", "exit db"], "finally must run on an exception"

    print("all coroutine checks passed")

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    verify()

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.